# Peak-Group Alignment of Mass Spectrometry Data (PGAlign)

Anji Deng, Yuyang Zhang, Qihuang Zhang\*

McGill University (corresponding author: <qihuang.zhang@mcgill.ca>)

Reference: Deng A, Zhang Y, Zhang Q. *PGAlign: peak group alignment of mass spectrometry data*. Under review at the *Journal of Proteome Research*, 2026.

This tutorial uses **PGAlign** to align the Week 5 spectra of the atherosclerosis regression
experiment onto Week 2, as in the manuscript, and returns them on the Week 2 m/z grid for
joint downstream analyses such as spatial segmentation.

We assume you have:

- the spectra of each time point as a CSV export (one row per spot, one column per m/z bin)
- the spatial coordinates of each spot as a second CSV
- the `pgalign` package installed (`pip install -e .` from the repository root)

---

## 0. Setup

In [ ]:
import os

import numpy as np
import pandas as pd
import scanpy as sc

import pgalign as pg
pg.__version__

## 1. Load MALDI data and create AnnData objects

Set `DataDir` below to the local folder where you have placed the four CSV files for
this tutorial (two weeks &times; spectra + region-spots). The mean spectra of the Maaike (atherosclerosis
regression) dataset are on Zenodo (<https://zenodo.org/records/20533948>); the per-spot spectra this tutorial reads
are available from the data owners upon reasonable request. See `CodeInPaper/Maaike/README.md` for details.

In [ ]:
# === CONFIGURE ME ===
DataDir = "data/Maaike"   # path to the Maaike Schuurman dataset CSVs
# ====================

weekpoint1 = "5 wk regression"
weekpoint2 = "2 wk regression"

groupshort = "pos"

if groupshort == "pos":
    group = "DHB pos"
else:
    group = "9AA neg"

In [ ]:
DataDir

In [ ]:
weekpoint1

In [ ]:
group

In [ ]:
MALDIdata1 = pd.read_csv("{DataDir}/{weekpoint} - {group} - All Spectra.csv".format(weekpoint = weekpoint1, DataDir = DataDir, group = group), sep =';') 
MALDIloc1 = pd.read_csv("{DataDir}/{weekpoint} - {group} - Region Spots.csv".format(weekpoint = weekpoint1, DataDir = DataDir, group = group), sep =';') 
MALDIdata2 = pd.read_csv("{DataDir}/{weekpoint} - {group} - All Spectra.csv".format(weekpoint = weekpoint2, DataDir = DataDir, group = group), sep =';') 
MALDIloc2 = pd.read_csv("{DataDir}/{weekpoint} - {group} - Region Spots.csv".format(weekpoint = weekpoint2, DataDir = DataDir, group = group), sep =';') 

In [ ]:
mz_value1 = pd.DataFrame(list(MALDIdata1.columns[1:]), index = list(MALDIdata1.columns[1:])).astype('float')
mz_value1 = mz_value1.rename(columns = {0:"m/z"})

mz_value2 = pd.DataFrame(list(MALDIdata2.columns[1:]), index = list(MALDIdata2.columns[1:])).astype('float')
mz_value2 = mz_value2.rename(columns = {0:"m/z"})

In [ ]:
mz_value1

In [ ]:
np.diff(mz_value1["m/z"])

In [ ]:
mz_value2

Create AnnData object for each MSI data

In [ ]:
MALDIdataAnn1 = sc.AnnData(X = MALDIdata1.iloc[:,1:], var = mz_value1, obs = MALDIloc1)
MALDIdataAnn2 = sc.AnnData(X = MALDIdata2.iloc[:,1:], var = mz_value2, obs = MALDIloc2)

In [ ]:
MALDIdataAnn1.obs["x"]

## 2. Coordinate formatting

PGAlign expects integer coordinates. The intensities are passed as they are: `pg.align`
normalizes each spot itself, on copies of the data.

In [ ]:
MALDIdataAnn1.obs = MALDIdataAnn1.obs.astype(int)
MALDIdataAnn2.obs = MALDIdataAnn2.obs.astype(int)

## 3. Align Week 5 onto Week 2

`pg.align` runs the four steps of the manuscript and the diagnostic for ambiguous alignments.
Its defaults are the settings of the atherosclerosis analysis: peaks called on the three-point
moving average with &alpha; = 0.9, a peak grouping parameter of 0.9, &delta; = 0 for the pairing and
&tau; = 0.1 for the diagnostic.

In [ ]:
res = pg.align(MALDIdataAnn1, MALDIdataAnn2)
print(res.summary())

On the DHB-positive data this reports 198 paired peak groups, 78 of them shifted and 20 flagged,
the counts given in the manuscript.

`res.shifts` lists each paired peak group with the Week 2 m/z range it occupies and its offset in
grid points:

In [ ]:
res.shifts.head()

`res.ambiguity` lists every shifted group with the score of the selected alignment and of leaving
the group where it was. A group is flagged when the two lie within &tau; of each other; flagged
groups deserve a look before downstream analysis.

In [ ]:
res.ambiguity[res.ambiguity["flagged"]]

## 4. Inside the four steps

The intermediate results stay available: `res.peaks` holds Steps 1 and 2, and `res.steps`
holds Steps 3 and 4.

**Step 1: Peak calling.** The mean spectrum of each dataset is smoothed with a three-point moving
average, and a point is called a peak when its smoothed intensity exceeds that of a neighbour by
more than the &alpha;-quantile of these differences. The number of peak points called in Week 5
and in Week 2:

In [ ]:
int(res.peaks.peakUnk.sum()), int(res.peaks.peakRef.sum())

**Step 2: Peak grouping.** The called peaks of both spectra are pooled into one sorted sequence
of m/z values, and the sequence is cut wherever two adjacent peaks lie farther apart than the
0.9 quantile of the adjacent distances (the peak grouping parameter, `percentile` in `pg.align`).
This is one joint partition of the pooled peaks, not a merge of two separately computed
partitions, and the remaining steps use these joint peak groups as their scaffold. A larger
`percentile` gives fewer, larger groups. The first joint peak group:

In [ ]:
res.peaks.jointcluster[0]

`save_clusterresults` writes the m/z range of each group to a CSV file:

In [ ]:
os.makedirs("output/example", exist_ok=True)
pg.save_clusterresults(res.peaks.jointcluster, "output/example/joint_peak_groups")

**Step 3: Peak group pairing.** A greedy algorithm on the (distance-penalised) similarity matrix
pairs unknown groups with reference groups. The first pairs, as (Week 5 group, Week 2 group):

In [ ]:
res.steps.align_group[:5]

**Step 4: Fine Alignment Assessment.** For each paired peak group, slide a window of &plusmn;4 grid points (four spacings of the m/z grid, not an m/z interval of 4) and pick the rigid integer-index shift that maximises the diagonal-mean Pearson correlation. A group already in the right place attains its maximum at the zero offset and is left unshifted.

In [ ]:
res.shifts["offset"].value_counts().sort_index()

## 5. The aligned dataset

`res.aligned` holds the Week 5 spectra on the Week 2 m/z grid, and `res.reference` the Week 2
spectra on the same bins and the same normalized scale, so the two can be analysed jointly, for
example by PCA, Harmony and clustering. Both are cut to the shorter of the two m/z axes.

In [ ]:
res.aligned.shape, res.reference.shape